# Classical Entailment Reasoning Agent

This notebook builds a symbolic forward-chaining reasoner. It does not use a language model to decide labels. The included English parser covers a deliberately bounded subset of controlled statements; unsupported constructions are reported rather than silently treated as successfully parsed. FOLIO contains broader first-order logic and requires richer parsing for full coverage.

The inference target is positive Horn entailment. A query is predicted `True` when it is derivable; otherwise it is `False` under the benchmarks' binary-label convention. This is not a complete open-world first-order theorem prover: absence of a derivation is not proof of logical negation.

In [57]:
# 1. Import Required Libraries
from __future__ import annotations

import json
import re
import time
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable, Optional

## 2. Load and Explore Datasets

The Hugging Face `datasets` package is optional. Set `LOAD_PUBLIC_DATASETS = True` to download public splits. Repository IDs and split names are configurable because dataset mirrors/configurations can change; inspect the printed schema before evaluating. No private test data is read here.

In [58]:
# Set to True in an environment with network access and `datasets` installed.
%pip install datasets
LOAD_PUBLIC_DATASETS = True
PUBLIC_DATASET_REPOS = {
    "RuleTaker": "jise/ruletaker",
    "CounterBench": "CounterBench/CounterBench",
    "FOLIO": "yale-nlp/FOLIO",
}
PUBLIC_SPLITS = {"RuleTaker": "test", "FOLIO": "validation"}
COUNTERBENCH_FILES = [
    "hf://datasets/CounterBench/CounterBench/data_balanced_alpha_V1.json",
    "hf://datasets/CounterBench/CounterBench/data_balanced_backdoor_V2.json",
]
public_datasets: dict[str, Any] = {}


def load_public_datasets() -> dict[str, Any]:
    try:
        from datasets import concatenate_datasets, load_dataset
    except ImportError as exc:
        raise RuntimeError("Install `datasets` to load Hugging Face datasets.") from exc

    loaded = {}
    for name, repo_id in PUBLIC_DATASET_REPOS.items():
        try:
            if name == "CounterBench":
                parts = [
                    load_dataset("json", data_files=path, split="train")
                    for path in COUNTERBENCH_FILES
                ]
                dataset = concatenate_datasets(parts)
            else:
                dataset = load_dataset(repo_id, split=PUBLIC_SPLITS[name])
            loaded[name] = dataset
            print(f"{name}: {len(dataset)} rows; columns={dataset.column_names}")
            if len(dataset):
                print("  example keys:", list(dataset[0].keys()))
        except Exception as exc:
            print(f"Could not load {name} ({repo_id}): {exc}")
            if name == "FOLIO":
                print("  FOLIO access may require accepting the dataset terms and authenticating with Hugging Face.")
            else:
                print("  Check the repository, split, and network access, then update the constants above.")
    return loaded


if LOAD_PUBLIC_DATASETS:
    public_datasets = load_public_datasets()
else:
    print("Dataset downloads disabled. Set LOAD_PUBLIC_DATASETS=True to inspect the public splits.")

Note: you may need to restart the kernel to use updated packages.
RuleTaker: 250 rows; columns=['context', 'statement', 'reasoning', 'depth', 'flag']
  example keys: ['context', 'statement', 'reasoning', 'depth', 'flag']
CounterBench: 1200 rows; columns=['question', 'given_info', 'answer', 'type', 'question_id', 'meta']
  example keys: ['question', 'given_info', 'answer', 'type', 'question_id', 'meta']
FOLIO: 203 rows; columns=['story_id', 'premises', 'premises-FOL', 'conclusion', 'conclusion-FOL', 'label', 'example_id']
  example keys: ['story_id', 'premises', 'premises-FOL', 'conclusion', 'conclusion-FOL', 'label', 'example_id']


In [59]:
from datasets import get_dataset_config_names

for repo_id in ("jise/ruletaker", "CounterBench/CounterBench"):
    try:
        print(repo_id, get_dataset_config_names(repo_id))
    except Exception as exc:
        print(repo_id, type(exc).__name__, exc)

jise/ruletaker ['default']
CounterBench/CounterBench ['default']


In [60]:
from datasets import load_dataset

ruletaker_test = load_dataset("jise/ruletaker", split="test")
print("RuleTaker test:", len(ruletaker_test), ruletaker_test.column_names)

counterbench_paths = [
    "hf://datasets/CounterBench/CounterBench/data_balanced_alpha_V1.json",
    "hf://datasets/CounterBench/CounterBench/data_balanced_backdoor_V2.json",
]
counterbench_parts = [load_dataset("json", data_files=path, split="train") for path in counterbench_paths]
print("CounterBench columns:", [dataset.column_names for dataset in counterbench_parts])
print("CounterBench row examples:", [dataset[0] for dataset in counterbench_parts])

RuleTaker test: 250 ['context', 'statement', 'reasoning', 'depth', 'flag']
CounterBench columns: [['question', 'given_info', 'answer', 'type', 'question_id', 'meta'], ['question', 'given_info', 'answer', 'type', 'question_id', 'meta']]
CounterBench row examples: [{'question': 'Would Lumbo occur if not Ziklo instead of Ziklo?', 'given_info': 'We know that Ziklo causes Blaf, Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo.', 'answer': 'no', 'type': 'basic', 'question_id': '0', 'meta': {'story_id': 'nonsense_0', 'rung': 3, 'query_type': 'det-counterfactual', 'model_id': 0, 'graph_id': 'graph5'}}, {'question': 'Would Lumbo occur if not Ziklo instead of Ziklo?', 'given_info': 'We know that Blaf causes Ziklo, Ziklo or Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo. Blaf~ Bern(0.2). We observed Trune', 'answer': 'yes', 'type': 'backdoor', 'question_id': '0', 'meta': {'story_id': 'nonsense_0', 'rung': 3, 'query_type': 'det-counterfactual', 'model_id': 0, 'graph_id': 'g

In [61]:
for index in range(5):
    row = ruletaker_test[index]
    print(f"Example {index + 1} ({row['flag']}): {row['statement']}\n{row['context']}\n")

Example 1 (False): The book volunteers.
Facts:
Liam volunteers.
The shop sings beautifully.
Liam volunteers.
Rules:
When liam volunteers, we can conclude liam hikes mountains.
Given that liam volunteers, it follows that liam sings beautifully.
Given that henry volunteers, it follows that the book sings beautifully.
Given that liam volunteers, it follows that liam sings beautifully.
If liam volunteers, then henry volunteers.
Based on liam sings beautifully, it is evident that henry volunteers.

Example 2 (False): The sky is fluffy.
Facts:
Lucy is a scientist.
Lucy is a scientist.
The mountain is fluffy.
Rules:
When lucy is a scientist, we can conclude the mountain is fluffy.
Based on the sky is fluffy, it is evident that the sky dances gracefully.
Assuming the stadium is a scientist, we can say the sky dances gracefully.
If the stadium is a scientist, then the mountain is a scientist.
When the stadium travels far, we can conclude the stadium is a scientist.
When the mountain is fluffy, 

## 3. Parse Knowledge Base: Facts and Rules

The parser handles simple property facts, binary relations, explicit negated facts, conjunctions, and universally quantified `If ... then ...` rules expressed in controlled English. Unsupported wording raises `ParseError`, allowing evaluation to count coverage instead of guessing.

In [62]:
class ParseError(ValueError):
    pass


UNIVERSAL_TERMS = {"someone", "somebody", "something", "anyone", "anybody", "everyone", "everybody", "they", "he", "she", "it"}


def _term(text: str) -> str:
    cleaned = re.sub(r"^(a|an|the)\s+", "", text.strip(), flags=re.IGNORECASE)
    normalized = re.sub(r"\s+", "_", cleaned.lower())
    return "?x" if cleaned.lower() in UNIVERSAL_TERMS else normalized


def _sentences(text: Any) -> list[str]:
    if isinstance(text, (list, tuple)):
        text = " ".join(str(part) for part in text)
    if not isinstance(text, str):
        raise ParseError(f"Expected text, got {type(text).__name__}")
    return [part.strip() for part in re.split(r"[.!\n]+", text) if part.strip()]


def parse_atom_clause(text: str) -> "Atom":
    clause = re.sub(r"[,;]+", " ", text.strip().rstrip(".?!"))
    clause = re.sub(r"\s+", " ", clause)
    negated = False
    if clause.lower().startswith("not "):
        negated = True
        clause = clause[4:].strip()
    if re.match(r"^(we know|given that|when |based on|it follows|would |could |does |did )", clause, re.IGNORECASE):
        raise ParseError(f"Unsupported natural-language construction: {text}")

    copula = re.match(r"^(.+?)\s+(is|are|was|were)\s+(.+)$", clause, flags=re.IGNORECASE)
    if copula:
        subject, _, complement = copula.groups()
        complement = complement.strip()
        if complement.lower().startswith("not "):
            negated = not negated
            complement = complement[4:].strip()
        complement = re.sub(r"^(a|an|the)\s+", "", complement, flags=re.IGNORECASE)
        predicate = re.sub(r"\s+", "_", complement.lower())
        if not predicate:
            raise ParseError(f"Missing predicate in: {text}")
        return Atom(predicate, (_term(subject),), negated)

    unary = re.match(r"^(?:(?:the|a|an)\s+)?([a-z][a-z-]*)\s+([a-z][a-z-]*)$", clause, flags=re.IGNORECASE)
    if unary:
        subject, predicate = unary.groups()
        return Atom(predicate.lower(), (_term(subject),), negated)

    relation = re.match(r"^(?:(?:the|a|an)\s+)?([a-z][a-z-]*)\s+([a-z][a-z-]*)\s+(.+)$", clause, flags=re.IGNORECASE)
    if relation:
        subject, predicate, object_term = relation.groups()
        return Atom(predicate.lower(), (_term(subject), _term(object_term)), negated)
    raise ParseError(f"Unsupported clause: {text}")


def _expand_conjunction(text: str) -> list[str]:
    pieces = [piece.strip() for piece in re.split(r"\s+and\s+", text, flags=re.IGNORECASE)]
    expanded = []
    frame = None
    for piece in pieces:
        if not re.search(r"\b(is|are|was|were)\b", piece, flags=re.IGNORECASE) and frame:
            piece = frame + piece
        expanded.append(piece)
        match = re.match(r"^(.+?\s+(?:is|are|was|were)\s+)", piece, flags=re.IGNORECASE)
        if match:
            frame = match.group(1)
    return expanded


def parse_knowledge_base(context: Any) -> tuple[list["Atom"], list["HornRule"], list[str]]:
    facts, rules, unsupported = [], [], []
    rule_patterns = (
        r"^if\s+(.+?)\s*,?\s+then\s+(.+)$",
        r"^when\s+(.+?),\s+we can conclude\s+(.+)$",
        r"^given that\s+(.+?),\s+it follows that\s+(.+)$",
        r"^assuming\s+(.+?),\s+we can say\s+(.+)$",
        r"^based on\s+(.+?),\s+it is evident that\s+(.+)$",
    )
    unsupported_pattern = re.compile(
        r"\b(all|each|every|everyone|everybody|no|neither|either|or|iff|unless|who|whom|whose|which)\b|if and only if",
        flags=re.IGNORECASE,
    )
    for sentence in _sentences(context):
        if sentence.strip().lower().rstrip(":") in {"facts", "rules"}:
            continue
        if unsupported_pattern.search(sentence):
            unsupported.append(f"{sentence}: quantified, relative, or disjunctive statement is outside the Horn parser")
            continue
        rule_match = next((re.match(pattern, sentence, flags=re.IGNORECASE) for pattern in rule_patterns
                           if re.match(pattern, sentence, flags=re.IGNORECASE)), None)
        try:
            if rule_match:
                body_text, head_text = rule_match.groups()
                body = tuple(parse_atom_clause(part) for part in _expand_conjunction(body_text))
                head_parts = _expand_conjunction(head_text)
                if len(head_parts) != 1:
                    raise ParseError("Conjunctive rule consequents are not supported")
                rules.append(HornRule(body, parse_atom_clause(head_parts[0])))
            else:
                parsed_facts = [parse_atom_clause(part) for part in _expand_conjunction(sentence)]
                if any(term.startswith("?") for fact in parsed_facts for term in fact.arguments):
                    raise ParseError("Universal variables are not allowed in ground facts")
                facts.extend(parsed_facts)
        except ParseError as exc:
            unsupported.append(f"{sentence}: {exc}")
    return facts, rules, unsupported


def parse_query(query: Any) -> "Atom":
    if isinstance(query, str) and "?" in query:
        raise ParseError("Interrogative question syntax is unsupported")
    sentences = _sentences(query)
    if len(sentences) != 1:
        raise ParseError("A query must contain exactly one statement")
    return parse_atom_clause(sentences[0])

## 4. Internal Representation of Facts and Rules

Atoms store a predicate, ordered terms, and an explicit sign. Terms beginning with `?` are variables. A Horn rule contains a conjunctive body and one head atom; rule variables are grounded by unification during inference.

In [63]:
import sys

LOGIC_MODULE_DIR = Path.cwd()
if not (LOGIC_MODULE_DIR / "logic_engine.py").exists():
    LOGIC_MODULE_DIR = LOGIC_MODULE_DIR / "submission"
sys.path.insert(0, str(LOGIC_MODULE_DIR))

from logic_engine import Atom, HornRule, KnowledgeBase, make_knowledge_base

## 5. Reasoning Engine Implementation

The engine repeatedly joins each rule body against currently known signed ground atoms, using first-order unification for variables. Newly instantiated heads are added until a fixed point is reached. The finite Herbrand domain is the constants already present in the facts; no generative or neural inference is involved.

In [64]:
from logic_engine import forward_chain, unify

## 6. Query Processing and Entailment Checking

Text entry points return a prediction only when the context and query are within the parser's supported fragment. This makes coverage measurable and prevents malformed or unsupported examples from being counted as negative entailments.

In [85]:
import importlib
import folio_horn_adapter

folio_horn_adapter = importlib.reload(folio_horn_adapter)
UnsupportedFormula = folio_horn_adapter.UnsupportedFormula
folio_entails = folio_horn_adapter.entails


def predict_from_text(context: Any, query_text: Any) -> dict[str, Any]:
    facts, rules, unsupported = parse_knowledge_base(context)
    if unsupported:
        return {"prediction": None, "status": "unsupported_context", "details": unsupported, "trace": []}
    try:
        query_atom = parse_query(query_text)
    except ParseError as exc:
        return {"prediction": None, "status": "unsupported_query", "details": [str(exc)], "trace": []}
    if any(term.startswith("?") for term in query_atom.arguments):
        return {"prediction": None, "status": "variable_query", "details": ["Queries must name a ground entity"], "trace": []}

    closure, trace = forward_chain(make_knowledge_base(facts, rules))
    return {
        "prediction": query_atom in closure,
        "status": "ok",
        "details": [],
        "trace": [f"{rule} ==> {conclusion}" for rule, conclusion in trace],
    }


FIELD_ALIASES = {
    "id": ("id", "ID", "example_id", "uid", "question_id"),
    "context": ("context", "facts", "premises", "story", "given_info"),
    "query": ("question", "query", "statement", "hypothesis", "conclusion"),
    "label": ("label", "answer", "gold_label", "entailment", "target", "flag"),
}


def _get_field(row: dict[str, Any], aliases: tuple[str, ...], default=None):
    lowered = {str(key).lower(): value for key, value in row.items()}
    for alias in aliases:
        if alias.lower() in lowered:
            return lowered[alias.lower()]
    return default


def normalize_label(value: Any) -> Optional[bool]:
    if isinstance(value, bool):
        return value
    if isinstance(value, (int, float)) and value in (0, 1):
        return bool(value)
    normalized = str(value).strip().lower()
    if normalized in {"true", "yes", "entailed", "entailment", "1"}:
        return True
    if normalized in {"false", "no", "not entailed", "refuted", "0"}:
        return False
    return None


def predict_folio_record(row: dict[str, Any]) -> dict[str, Any]:
    example_id = _get_field(row, ("example_id", "id"))
    gold = normalize_label(_get_field(row, ("label",)))
    try:
        prediction, trace = folio_entails(row["premises-FOL"], row["conclusion-FOL"])
    except UnsupportedFormula as exc:
        return {"id": example_id, "gold": gold, "prediction": None,
                "status": "unsupported_formula", "details": [str(exc)], "trace": []}
    return {"id": example_id, "gold": gold, "prediction": prediction,
            "status": "ok", "details": [], "trace": trace}


def predict_record(row: dict[str, Any]) -> dict[str, Any]:
    if "premises-FOL" in row and "conclusion-FOL" in row:
        formal_result = predict_folio_record(row)
        if formal_result["status"] == "ok":
            return formal_result
        english_result = predict_from_text(row.get("premises", ""), row.get("conclusion", ""))
        if english_result["status"] == "ok":
            english_result.update({"id": formal_result["id"], "gold": formal_result["gold"]})
            english_result["details"] = ["Used controlled-English fallback"]
            return english_result
        formal_result["details"].append(f"English fallback: {english_result['status']}")
        return formal_result

    example_id = _get_field(row, FIELD_ALIASES["id"])
    context = _get_field(row, FIELD_ALIASES["context"])
    query = _get_field(row, FIELD_ALIASES["query"])
    if context is None or query is None:
        return {"id": example_id, "gold": normalize_label(_get_field(row, FIELD_ALIASES["label"])),
                "prediction": None, "status": "unrecognized_schema", "details": [list(row.keys())], "trace": []}
    result = predict_from_text(context, query)
    result.update({"id": example_id, "gold": normalize_label(_get_field(row, FIELD_ALIASES["label"]))})
    return result

## 7. Batch Inference on Public Dataset

Each row is passed through the same parser and reasoner. The batch records predictions, labels, status, and derivation traces. Unsupported rows remain explicit and are excluded from accuracy while still contributing to coverage counts.

In [86]:
def run_batch(rows: Iterable[dict[str, Any]], limit: Optional[int] = None) -> tuple[list[dict[str, Any]], float]:
    start = time.perf_counter()
    results = []
    for index, row in enumerate(rows):
        if limit is not None and index >= limit:
            break
        results.append(predict_record(dict(row)))
    return results, time.perf_counter() - start


demo_rows = [
    {
        "id": "demo-chain-true",
        "context": "Bob is young. Bob is kind. If someone is young, then they are smart. If someone is smart and kind, then they are successful.",
        "question": "Bob is successful.",
        "label": True,
    },
    {
        "id": "demo-not-derived",
        "context": "Bob is young. If someone is young, then they are smart.",
        "question": "Bob is successful.",
        "label": False,
    },
]

demo_results, demo_elapsed = run_batch(demo_rows)
assert all(result["status"] == "ok" for result in demo_results), demo_results
assert [result["prediction"] for result in demo_results] == [True, False], demo_results
print(f"Synthetic smoke test: {len(demo_results)} examples, {demo_elapsed:.6f}s")
print("Derived trace:", demo_results[0]["trace"])

public_batch_results = {}
for dataset_name, dataset in public_datasets.items():
    results, elapsed = run_batch(dataset)
    public_batch_results[dataset_name] = {"results": results, "elapsed_seconds": elapsed}
    supported = [result for result in results if result["status"] == "ok"]
    print(f"{dataset_name}: {len(supported)}/{len(results)} supported; {elapsed:.3f}s")

Synthetic smoke test: 2 examples, 0.000721s
Derived trace: ['young(?x) -> smart(?x) ==> smart(bob)', 'smart(?x), kind(?x) -> successful(?x) ==> successful(bob)']
RuleTaker: 250/250 supported; 0.238s
CounterBench: 0/1200 supported; 0.181s
FOLIO: 55/203 supported; 0.072s


## 8. Evaluate Accuracy and Inference Time

Accuracy is calculated only for rows that both parse successfully and have a recognizable gold label. Coverage reports what fraction of rows the parser/reasoner handled. Timing includes parsing and inference for the full batch; public-set metrics are printed only when public datasets were actually loaded.

In [87]:
def summarize_results(results: list[dict[str, Any]], elapsed_seconds: float) -> dict[str, Any]:
    evaluated = [item for item in results if item["status"] == "ok" and item["gold"] is not None]
    correct = sum(item["prediction"] == item["gold"] for item in evaluated)
    summary = {
        "rows": len(results),
        "supported": sum(item["status"] == "ok" for item in results),
        "labeled_supported": len(evaluated),
        "correct": correct,
        "accuracy": correct / len(evaluated) if evaluated else None,
        "coverage": sum(item["status"] == "ok" for item in results) / len(results) if results else 0.0,
        "elapsed_seconds": elapsed_seconds,
        "seconds_per_row": elapsed_seconds / len(results) if results else 0.0,
    }
    return summary


if public_batch_results:
    public_metrics = {}
    for dataset_name, batch in public_batch_results.items():
        metrics = summarize_results(batch["results"], batch["elapsed_seconds"])
        public_metrics[dataset_name] = metrics
        accuracy = "n/a" if metrics["accuracy"] is None else f"{metrics['accuracy']:.2%}"
        print(
            f"{dataset_name}: accuracy={accuracy}, coverage={metrics['coverage']:.2%}, "
            f"supported={metrics['supported']}/{metrics['rows']}, "
            f"inference_time={metrics['elapsed_seconds']:.3f}s"
        )
else:
    demo_metrics = summarize_results(demo_results, demo_elapsed)
    print("No public dataset was loaded; these are synthetic smoke-test metrics, not benchmark accuracy.")
    print(f"Synthetic accuracy={demo_metrics['accuracy']:.2%}; inference_time={demo_metrics['elapsed_seconds']:.6f}s")

RuleTaker: accuracy=97.60%, coverage=100.00%, supported=250/250, inference_time=0.238s
CounterBench: accuracy=n/a, coverage=0.00%, supported=0/1200, inference_time=0.181s
FOLIO: accuracy=83.33%, coverage=27.09%, supported=55/203, inference_time=0.072s


## 9. Generate Submission File

Provide a CSV of evaluation examples with ID, context, query, and optionally a label column. The notebook does not locate or open a private test file automatically. The writer refuses to emit a partial submission if any row is unsupported, preserving one prediction per supplied ID.

In [88]:
import csv
import tempfile


def generate_submission(rows: Iterable[dict[str, Any]], output_path: str | Path) -> Path:
    rows = list(rows)
    results = [predict_record(dict(row)) for row in rows]
    unsupported = [result for result in results if result["status"] != "ok" or result["prediction"] is None]
    if unsupported:
        statuses = defaultdict(int)
        for item in unsupported:
            statuses[item["status"]] += 1
        raise ValueError(f"Cannot generate complete submission; unsupported rows by status: {dict(statuses)}")

    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    with output.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=["ID", "Label"])
        writer.writeheader()
        for original, result in zip(rows, results):
            example_id = _get_field(dict(original), FIELD_ALIASES["id"])
            if example_id is None:
                raise ValueError("Every submission row must include an ID")
            writer.writerow({"ID": example_id, "Label": str(result["prediction"])})
    return output


def generate_submission_from_csv(input_path: str | Path, output_path: str | Path = "submission/submission.csv") -> Path:
    with Path(input_path).open(newline="", encoding="utf-8") as file:
        rows = list(csv.DictReader(file))
    return generate_submission(rows, output_path)


# Safe writer smoke test in a temporary directory; it does not create a submission file.
with tempfile.TemporaryDirectory() as temp_dir:
    smoke_path = generate_submission(demo_rows, Path(temp_dir) / "submission.csv")
    with smoke_path.open(newline="", encoding="utf-8") as file:
        smoke_rows = list(csv.DictReader(file))
    assert [row["Label"] for row in smoke_rows] == ["True", "False"]
    print(f"Submission writer smoke test passed ({len(smoke_rows)} rows).")

# For an authorized evaluation CSV, run explicitly, for example:
# generate_submission_from_csv("submission/evaluation_input.csv", "submission/submission.csv")

Submission writer smoke test passed (2 rows).


In [84]:
counterbench = public_datasets["CounterBench"]
print("CounterBench type/label counts:", Counter((row["type"], row["answer"]) for row in counterbench))
for category in sorted({row["type"] for row in counterbench}):
    sample = next(row for row in counterbench if row["type"] == category)
    print(f"\nCounterBench {category} example:\nContext: {sample['given_info']}\nQuestion: {sample['question']}\nAnswer: {sample['answer']}")

CounterBench type/label counts: Counter({('basic', 'no'): 125, ('basic', 'yes'): 125, ('conditional', 'yes'): 125, ('conditional', 'no'): 125, ('joint', 'yes'): 125, ('joint', 'no'): 125, ('nested', 'yes'): 125, ('nested', 'no'): 125, ('backdoor', 'no'): 105, ('backdoor', 'yes'): 95})

CounterBench backdoor example:
Context: We know that Blaf causes Ziklo, Ziklo or Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo. Blaf~ Bern(0.2). We observed Trune
Question: Would Lumbo occur if not Ziklo instead of Ziklo?
Answer: yes

CounterBench basic example:
Context: We know that Ziklo causes Blaf, Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo.
Question: Would Lumbo occur if not Ziklo instead of Ziklo?
Answer: no

CounterBench conditional example:
Context: We know that Ziklo causes not Blaf, Blaf and Trune together cause Vork, Vork causes Lumbo. We observed Trune
Question: Would Lumbo occur if not Ziklo instead of Ziklo?
Answer: yes

CounterBench joint example:
Context: We

In [74]:
from collections import Counter

for dataset_name, batch in public_batch_results.items():
    results = batch["results"]
    status_counts = Counter(result["status"] for result in results)
    print(f"\n{dataset_name} status counts:", status_counts)
    reasons = Counter(
        detail.split(":", 1)[-1].strip()[:100]
        for result in results if result["status"] != "ok"
        for detail in result.get("details", [])
    )
    print("Common unsupported reasons:", reasons.most_common(8))

if folio_dataset is not None:
    print("\nFOLIO formal example:")
    print({key: folio_dataset[0][key] for key in ("premises-FOL", "conclusion-FOL", "label")})


RuleTaker status counts: Counter({'ok': 250})
Common unsupported reasons: []

CounterBench status counts: Counter({'unsupported_context': 1200})
Common unsupported reasons: [('quantified, relative, or disjunctive statement is outside the Horn parser', 365), ('Unsupported clause: 6)', 80), ('Unsupported clause: 4)', 65), ('Unsupported clause: 2)', 45), ('Unsupported natural-language construction: We know that Ziklo causes Blaf, Blaf causes Trune, Trune ', 42), ('Unsupported natural-language construction: We know that Glent causes Razz, Razz causes Pex, Pex caus', 42), ('Unsupported natural-language construction: We know that Praf causes Vank, Vank causes Scud, Scud cau', 42), ('Unsupported natural-language construction: We know that Fizo causes Blorn, Blorn causes Plim, Plim c', 42)]

FOLIO status counts: Counter({'unsupported_context': 172, 'ok': 29, 'unsupported_query': 2})
Common unsupported reasons: [('quantified, relative, or disjunctive statement is outside the Horn parser', 475)